In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import torchinfo
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

import helper_utils


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


## Hyperparameters

In [3]:
BATCH_SIZE = 32

## Datasets and DataLoaders

### Dataset Transforms

In [4]:
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

### Train, Test and Validation Datasets and DataLoaders

In [5]:
train_dir = os.path.join("data", "train")
test_dir = os.path.join("data", "test")
val_dir = os.path.join("data", "validation")

train_dataset = ImageFolder(root = train_dir, transform = None)
val_dataset = ImageFolder(root = val_dir, transform = None)
test_dataset = ImageFolder(root = test_dir, transform = None)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [6]:
classes = train_dataset.classes

num_classes = len(classes)

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

Classes: ['dress', 'hat', 'longsleeve', 'outwear', 'pants', 'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
Number of classes: 10


In [9]:
class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expansion_factor, shortcut = None):
        super(InvertedResidualBlock, self).__init__()

        hidden_dim = in_channels * expansion_factor



        # 1 x 1 pointwise convolution to expand the number of channels
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels =  in_channels,
                      out_channels = hidden_dim,
                      kernel_size = 1, 
                      stride = stride,
                      padding = 0,
                      bias = False
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 3 x 3 depthwise convolution
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = hidden_dim,
                      kernel_size = 3,
                      stride = stride,
                      padding = 1,
                      bias = False,
                      groups = hidden_dim
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 1 x 1 pointwise convolution to project back to the original number of channels
        # Bottleneck
        self.project = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = out_channels,
                      kernel_size = 1,
                      bias = False),
            nn.BatchNorm2d(out_channels)
        )

        # Optional shortcut connection for residual learning
        self.shortcut = shortcut



    def forward(self, x):

        skip = x

        out = self.expand(x)
        out = self.depthwise(out)
        out = self.project(out)

        if self.shortcut is not None:
            skip = self.shortcut(x)


        out = out + skip

        return F.relu(out)



In [10]:
# --- Verification ---
# Define parameters for a sample block instance
batch_size=32
in_ch = 16 # Input channels
out_ch = 16 # Output channels (same for stride=1)
stride = 1
exp_factor = 3 # Expansion factor
img_size = 32 # Input image height/width

# Instantiate the block
block = InvertedResidualBlock(
    in_channels=in_ch,
    out_channels=out_ch,
    stride=stride,
    expansion_factor=exp_factor,
)

# Define the input tensor shape
input_size = (batch_size, in_ch, img_size, img_size)

# Configuration for torchinfo summary
config = {
    "input_size": input_size,
    "attr_names": ["input_size", "output_size", "num_params"],
    "col_names_display": ["Input Shape ", "Output Shape", "Param #"],
    "depth": 3 # Show layers up to 3 levels deep
}

# Generate the summary
summary = torchinfo.summary(
    model=block,
    input_size=config["input_size"],
    col_names=config["attr_names"],
    depth=config["depth"]
)

# Display the formatted summary
print("--- Block Summary (Stride=1, Same Channels) ---\n")
helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Block Summary (Stride=1, Same Channels) ---



Layer (type (var_name):depth-idx),Input Shape,Output Shape,Param #
InvertedResidualBlock (InvertedResidualBlock),"[32, 16, 32, 32]","[32, 16, 32, 32]",--
Sequential (expand): 1-1,"[32, 16, 32, 32]","[32, 48, 32, 32]",--
Conv2d (0): 2-1,"[32, 16, 32, 32]","[32, 48, 32, 32]",768
BatchNorm2d (1): 2-2,"[32, 48, 32, 32]","[32, 48, 32, 32]",96
ReLU (2): 2-3,"[32, 48, 32, 32]","[32, 48, 32, 32]",--
Sequential (depthwise): 1-2,"[32, 48, 32, 32]","[32, 48, 32, 32]",--
Conv2d (0): 2-4,"[32, 48, 32, 32]","[32, 48, 32, 32]",432
BatchNorm2d (1): 2-5,"[32, 48, 32, 32]","[32, 48, 32, 32]",96
ReLU (2): 2-6,"[32, 48, 32, 32]","[32, 48, 32, 32]",--
Sequential (project): 1-3,"[32, 48, 32, 32]","[32, 16, 32, 32]",--


In [ ]:
class MobileNetBackbone(nn.Module):
    def __init__(self, num_classes):

        super(MobileNetBackbone, self).__init__()

        self.num_classes = num_classes

        self.stem = nn.Sequential(
           nn.Conv2d(in_channels = 3, out_channels = 16, stride = 2, padding = 1, bias = False),
           nn.BatchNorm2d(16),
           nn.ReLU(inplace = True)
        )

    



    def _make_block(self, in_channels, out_channels, stride, expansion_factor):

        condition = (stride !=1) or (in_channels != out_channels)

        if condition:
            shortcut = None

        else:
            shortcut = nn.Sequential(
                nn.Conv2d(in_channels = in_channels,
                          out_channels = out_channels,
                          stride = stride,
                          padding = 0,
                          bias = False),
                nn.BatchNorm2d(out_channels)
            )